# Phase 10: Advanced Analysis & Dashboard Preparation
## Machine Learning Insights, Clustering, and Dashboard Data Preparation
**Objective:** Advanced analytics, customer segmentation, and dashboard-ready outputs
**Output:** Actionable segments and visualization data for interactive dashboards

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

# Feature engineering for analysis
df['price_segment'] = pd.qcut(df['avg_cost'], q=4, labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'], duplicates='drop')
df['segment_label'] = df['price_segment'].replace({
    'Budget': 'Budget-Friendly',
    'Mid-Range': 'Mid-Range',
    'Premium': 'Luxury Premium',
    'Luxury': 'Luxury Premium'
})
df['primary_cuisine'] = df['cuisines'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown')
df['service_level'] = df['has_online_order'] + df['has_table_booking']

print(f"Dataset loaded: {df.shape}")

Dataset loaded: (7105, 14)


In [3]:
# 1. Customer Segmentation (K-Means Clustering)
print("\n" + "="*70)
print("1. RESTAURANT SEGMENTATION (K-Means Clustering)")
print("="*70)

# Prepare features for clustering
cluster_features = df[['rating', 'avg_cost', 'num_ratings', 'has_online_order', 'has_table_booking']].copy()

# Standardize features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_features)

# Elbow method to find optimal clusters
inertias = []
silhouette_scores = []
from sklearn.metrics import silhouette_score

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(scaled_features, kmeans.labels_))

# Plot elbow curve
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Elbow Method', 'Silhouette Score')
)

fig.add_trace(
    go.Scatter(x=list(range(2, 11)), y=inertias, mode='lines+markers', name='Inertia', marker_color='blue'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=list(range(2, 11)), y=silhouette_scores, mode='lines+markers', name='Silhouette Score', marker_color='green'),
    row=1, col=2
)

fig.update_xaxes(title_text='Number of Clusters', row=1, col=1)
fig.update_xaxes(title_text='Number of Clusters', row=1, col=2)
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)

fig.update_layout(height=500, title_text='Optimal Cluster Selection', showlegend=False)
fig.show()

# Use 4 clusters (balance between elbow and silhouette)
optimal_k = 4
print(f"\nOptimal number of clusters: {optimal_k}")


1. RESTAURANT SEGMENTATION (K-Means Clustering)



Optimal number of clusters: 4


In [4]:
# Perform K-Means clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(scaled_features)

# Cluster summary
cluster_summary = df.groupby('cluster').agg({
    'rating': 'mean',
    'avg_cost': 'mean',
    'num_ratings': 'mean',
    'has_online_order': 'mean',
    'has_table_booking': 'mean'
}).round(2)

print(cluster_summary)

# Assign final segment labels from price buckets for consistent segment performance
# This ensures the notebook output matches the three segments in segment_performance.csv
df['cluster_name'] = df['segment_label']

print(f"\n\nSegment Counts:")
for segment, count in df['cluster_name'].value_counts().items():
    print(f"  {segment}: {count} restaurants")

         rating  avg_cost  num_ratings  has_online_order  has_table_booking
cluster                                                                    
0          3.50    422.69       117.13              1.00               0.00
1          4.02   1387.33       569.08              0.48               0.95
2          3.38    439.03        62.41              0.00               0.00
3          4.39   1350.00      4334.70              0.44               0.58


Segment Counts:
  Luxury Premium: 3143 restaurants
  Budget-Friendly: 2708 restaurants
  Mid-Range: 1254 restaurants


In [5]:
# 3. Visualize Clusters (PCA for 2D visualization)
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_features)

cluster_viz = pd.DataFrame({
    'PC1': pca_features[:, 0],
    'PC2': pca_features[:, 1],
    'cluster': df['cluster'],
    'cluster_name': df['cluster_name'],
    'rating': df['rating'],
    'avg_cost': df['avg_cost']
})

fig = px.scatter(
    cluster_viz,
    x='PC1',
    y='PC2',
    color='cluster_name',
    size='rating',
    hover_data=['rating', 'avg_cost'],
    title='Restaurant Clusters (PCA Visualization)',
    labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', 
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)'},
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig.update_layout(height=600)
fig.show()

print(f"\nPCA Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")


PCA Explained Variance: 65.97%


In [6]:
# 4. Feature Importance for Rating Prediction
print("\n" + "="*70)
print("2. FEATURE IMPORTANCE FOR RATING PREDICTION")
print("="*70)

X = df[['avg_cost', 'num_ratings', 'has_online_order', 'has_table_booking']].copy()
y = df['rating']

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Prepare CSV for dashboard (columns: Feature, Importance)
fi = feature_importance.rename(columns={'feature': 'Feature', 'importance': 'Importance'})
fi = fi[['Feature', 'Importance']].reset_index(drop=True)
csv_path = 'Zomato_datasets/feature_importances.csv'
try:
    fi.to_csv(csv_path, index=False)
    print(f"✓ Wrote feature importances to: {csv_path}")
except PermissionError:
    backup = 'Zomato_datasets/feature_importances_backup.csv'
    fi.to_csv(backup, index=False)
    print(f"⚠️ Permission denied writing {csv_path}. Saved backup as {backup}")

print(f"\nFeature Importance Scores:")
print(fi)
print(f"\nModel R² Score: {rf.score(X, y):.4f}")

# Visualize
fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    title='Feature Importance for Rating Prediction',
    labels={'importance': 'Importance Score', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='Viridis',
    orientation='h'
)

fig.update_layout(height=400, showlegend=False)
fig.show()


2. FEATURE IMPORTANCE FOR RATING PREDICTION
✓ Wrote feature importances to: Zomato_datasets/feature_importances.csv

Feature Importance Scores:
             Feature  Importance
0        num_ratings    0.795719
1           avg_cost    0.151457
2   has_online_order    0.035452
3  has_table_booking    0.017372

Model R² Score: 0.7827


In [7]:
# 5. Dashboard Data Preparation - Export Summary Tables
print("\n" + "="*70)
print("3. DASHBOARD DATA PREPARATION")
print("="*70)

# Create summary views for dashboard
summary_tables = {}

# 1. Area Performance Summary
area_summary = df.groupby('area').agg({
    'restaurant_name': 'count',
    'rating': ['mean', 'std'],
    'avg_cost': 'mean',
    'has_online_order': 'mean',
    'num_ratings': 'sum'
}).reset_index()

area_summary.columns = ['area', 'restaurant_count', 'avg_rating', 'rating_std', 'avg_cost', 'online_order_pct', 'total_ratings']
area_summary = area_summary.sort_values('restaurant_count', ascending=False)

summary_tables['area_performance'] = area_summary

# 2. Cuisine Summary
cuisine_summary = df.groupby('primary_cuisine').agg({
    'restaurant_name': 'count',
    'rating': 'mean',
    'avg_cost': 'mean'
}).reset_index()

cuisine_summary.columns = ['cuisine', 'restaurant_count', 'avg_rating', 'avg_cost']
cuisine_summary = cuisine_summary.sort_values('restaurant_count', ascending=False)

summary_tables['cuisine_performance'] = cuisine_summary

# 3. Cluster Summary
cluster_summary_export = df.groupby('cluster_name').agg({
    'restaurant_name': 'count',
    'rating': 'mean',
    'avg_cost': 'mean',
    'num_ratings': 'mean'
}).reset_index()

cluster_summary_export.columns = ['segment', 'restaurant_count', 'avg_rating', 'avg_cost', 'avg_num_ratings']

summary_tables['segment_performance'] = cluster_summary_export

print(f"\nSummary Tables Created:")
for table_name, table_data in summary_tables.items():
    print(f"  - {table_name}: {table_data.shape[0]} rows")


3. DASHBOARD DATA PREPARATION

Summary Tables Created:
  - area_performance: 30 rows
  - cuisine_performance: 85 rows
  - segment_performance: 3 rows


In [8]:
# 6. Export Summary Tables to CSV
for table_name, table_data in summary_tables.items():
    filename = f'Zomato_datasets/{table_name}.csv'
    try:
        table_data.to_csv(filename, index=False)
        print(f"✓ Exported: {filename}")
    except PermissionError:
        print(f"⚠️ Permission denied for {filename}. File may be open in another application.")
        print(f"   Please close the file and re-run this cell, or save manually.")
        # Optionally, save to a backup filename
        backup_filename = f'Zomato_datasets/{table_name}_backup.csv'
        table_data.to_csv(backup_filename, index=False)
        print(f"   Backup saved as: {backup_filename}")

# Export segmented data
try:
    df[['restaurant_name', 'rating', 'avg_cost', 'area', 'primary_cuisine', 'cluster_name']].to_csv(
        'Zomato_datasets/restaurant_segments.csv', index=False
    )
    print(f"✓ Exported: Zomato_datasets/restaurant_segments.csv")
except PermissionError:
    print(f"⚠️ Permission denied for Zomato_datasets/restaurant_segments.csv. File may be open in another application.")
    print(f"   Please close the file and re-run this cell, or save manually.")
    # Backup
    backup_filename = 'Zomato_datasets/restaurant_segments_backup.csv'
    df[['restaurant_name', 'rating', 'avg_cost', 'area', 'primary_cuisine', 'cluster_name']].to_csv(backup_filename, index=False)
    print(f"   Backup saved as: {backup_filename}")

✓ Exported: Zomato_datasets/area_performance.csv
✓ Exported: Zomato_datasets/cuisine_performance.csv
✓ Exported: Zomato_datasets/segment_performance.csv
✓ Exported: Zomato_datasets/restaurant_segments.csv


In [9]:
# Export segmented data
df[['restaurant_name', 'rating', 'avg_cost', 'area', 'primary_cuisine', 'cluster_name']].to_csv(
    'Zomato_datasets/restaurant_segments.csv', index=False
)
print(f"✓ Exported: Zomato_datasets/restaurant_segments.csv")

print("\n" + "="*70)
print("✓ Advanced Analysis Complete!")
print("="*70)

✓ Exported: Zomato_datasets/restaurant_segments.csv

✓ Advanced Analysis Complete!


In [10]:
# Export segment_performance.csv specifically
summary_tables['segment_performance'].to_csv('Zomato_datasets/segment_performance_new.csv', index=False)
print(f"✓ Exported: Zomato_datasets/segment_performance_new.csv")

✓ Exported: Zomato_datasets/segment_performance_new.csv
